# 03 - Comparacion depth/child tuning

Compara los resultados de `P04 T03` y `P02 T02` ajustando solo `max_depth` y `min_child_weight`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "roadmap.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto.")


project_root = find_project_root()
results_dir = project_root / "54. Tuning de XGBoost" / "results"


In [2]:
summary_paths = [
    results_dir / "p04_t03_depth_child_summary.csv",
    results_dir / "p02_t02_depth_child_summary.csv",
]
missing = [path for path in summary_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"Faltan summaries. Ejecutar primero notebooks 01 y 02: {missing}")

comparison = pd.concat([pd.read_csv(path) for path in summary_paths], ignore_index=True)
comparison = comparison.sort_values(["ams_mean", "ams_std", "fit_seconds_mean"], ascending=[False, True, True]).reset_index(drop=True)
comparison_path = results_dir / "03_comparacion_depth_child.csv"
comparison.to_csv(comparison_path, index=False)

display(
    comparison[
        [
            "pipeline",
            "base_config_id",
            "depth_config_id",
            "learning_rate",
            "n_estimators",
            "max_depth",
            "min_child_weight",
            "ams_mean",
            "ams_std",
            "auc_mean",
            "auc_std",
            "threshold_median",
            "signal_rate_mean",
            "precision_at_ams_mean",
            "recall_at_ams_mean",
            "f1_at_ams_mean",
            "fit_seconds_mean",
        ]
    ].round(6)
)
print(f"Comparacion guardada en: {comparison_path}")


,pipeline,base_config_id,depth_config_id,learning_rate,n_estimators,max_depth,min_child_weight,ams_mean,ams_std,auc_mean,auc_std,threshold_median,signal_rate_mean,precision_at_ams_mean,recall_at_ams_mean,f1_at_ams_mean,fit_seconds_mean
0,P04,T03,D03,0.03,800,5,5,1.627843,0.093675,0.912033,0.001501,0.894817,0.146072,0.933044,0.396489,0.553305,7.071923
1,P04,T03,D06,0.03,800,5,10,1.617565,0.093741,0.911967,0.001488,0.872274,0.147940,0.931009,0.401135,0.558586,7.050740
2,P04,T03,D05,0.03,800,4,10,1.604187,0.088608,0.910354,0.001358,0.875425,0.146388,0.929932,0.396023,0.551936,6.068838
3,P04,T03,D02,0.03,800,4,5,1.598522,0.090340,0.910359,0.001323,0.887252,0.141820,0.932632,0.384874,0.541846,5.865558
4,P04,T03,D04,0.03,800,4,3,1.597762,0.084230,0.910343,0.001375,0.851559,0.153176,0.925503,0.412679,0.567872,5.857989
5,P02,T02,D02,0.05,500,4,5,1.589625,0.070211,0.908947,0.001559,0.870340,0.144780,0.930469,0.392439,0.550092,3.103278
6,P02,T02,D05,0.05,500,4,10,1.584463,0.067291,0.908908,0.001442,0.887343,0.143344,0.930502,0.388085,0.544748,2.555823
7,P04,T03,D01,0.03,800,3,5,1.555152,0.059580,0.906829,0.001449,0.875340,0.139464,0.928916,0.377707,0.535797,5.365167
8,P02,T02,D01,0.05,500,3,5,1.536479,0.049911,0.905413,0.001592,0.865655,0.144548,0.924863,0.389951,0.548018,2.586789


Comparacion guardada en: /home/santig14/fing/taa/1_TallerAA/54. Tuning de XGBoost/results/03_comparacion_depth_child.csv


## Decision

Elegir por balance: AMS medio, AMS std, tiempo, y threshold/signal rate razonables. Si la mejora de AMS es chica pero sube mucho la varianza o el tiempo, no conviene priorizarla.